In [1]:
import numpy as np
from scipy.stats import norm
from scipy.special import factorial
import pandas as pd

class MertonJumpModel:
    def __init__(self, S0, K, T, r, sigma, mu_j, delta_j, lambda_j):
        """
        Merton Jump Diffusion Model
        S0: Initial stock price
        K: Strike price
        T: Time to maturity
        r: Risk-free rate
        sigma: Volatility
        mu_j: Mean jump size (log-normal)
        delta_j: Standard deviation of jump size
        lambda_j: Jump intensity (expected number of jumps per year)
        """
        self.S0 = S0
        self.K = K
        self.T = T
        self.r = r
        self.sigma = sigma
        self.mu_j = mu_j
        self.delta_j = delta_j
        self.lambda_j = lambda_j

    def price_option(self, option_type='call', n_jumps_max=50):
        """
        Price European option using Merton model
        Sum over possible number of jump realizations
        """
        price = 0

        for n in range(n_jumps_max):
            # Probability of n jumps
            prob_n_jumps = np.exp(-self.lambda_j * self.T) * (self.lambda_j * self.T)**n / factorial(n)

            # Adjusted parameters for n jumps
            sigma_n = np.sqrt(self.sigma**2 + n * self.delta_j**2 / self.T)
            mu_n = self.r - self.lambda_j * (np.exp(self.mu_j + 0.5 * self.delta_j**2) - 1)

            # Black-Scholes price with adjusted parameters
            d1 = (np.log(self.S0 / self.K) + (mu_n + 0.5 * sigma_n**2) * self.T) / (sigma_n * np.sqrt(self.T))
            d2 = d1 - sigma_n * np.sqrt(self.T)

            if option_type == 'call':
                bs_price = self.S0 * norm.cdf(d1) - self.K * np.exp(-self.r * self.T) * norm.cdf(d2)
            else:  # put
                bs_price = self.K * np.exp(-self.r * self.T) * norm.cdf(-d2) - self.S0 * norm.cdf(-d1)

            price += prob_n_jumps * bs_price

        return price

    def delta(self, option_type='call', h=0.01, n_jumps_max=50):
        """Calculate delta using finite difference"""
        S_up = self.S0 + h
        S_down = self.S0 - h

        model_up = MertonJumpModel(S_up, self.K, self.T, self.r, self.sigma, self.mu_j, self.delta_j, self.lambda_j)
        model_down = MertonJumpModel(S_down, self.K, self.T, self.r, self.sigma, self.mu_j, self.delta_j, self.lambda_j)

        price_up = model_up.price_option(option_type, n_jumps_max)
        price_down = model_down.price_option(option_type, n_jumps_max)

        return (price_up - price_down) / (2 * h)

    def gamma(self, option_type='call', h=0.01, n_jumps_max=50):
        """Calculate gamma using finite difference"""
        S_up = self.S0 + h
        S_down = self.S0 - h

        model_up = MertonJumpModel(S_up, self.K, self.T, self.r, self.sigma, self.mu_j, self.delta_j, self.lambda_j)
        model_center = MertonJumpModel(self.S0, self.K, self.T, self.r, self.sigma, self.mu_j, self.delta_j, self.lambda_j)
        model_down = MertonJumpModel(S_down, self.K, self.T, self.r, self.sigma, self.mu_j, self.delta_j, self.lambda_j)

        price_up = model_up.price_option(option_type, n_jumps_max)
        price_center = model_center.price_option(option_type, n_jumps_max)
        price_down = model_down.price_option(option_type, n_jumps_max)

        return (price_up - 2 * price_center + price_down) / (h ** 2)



In [2]:
# Parameters
S0 = 100  # ATM: Strike = Spot
K = 100
T = 1  # 1 year
r = 0.05  # Risk-free rate
sigma = 0.2  # Base volatility
mu_j = -0.5  # Mean jump size
delta_j = 0.22  # Jump volatility

# Question 8: lambda = 0.75
print("=" * 60)
print("Question 8: Jump Intensity λ = 0.75")
print("=" * 60)
model_q8 = MertonJumpModel(S0, K, T, r, sigma, mu_j, delta_j, 0.75)
call_price_q8 = model_q8.price_option('call')
put_price_q8 = model_q8.price_option('put')
call_delta_q8 = model_q8.delta('call')
call_gamma_q8 = model_q8.gamma('call')
put_delta_q8 = model_q8.delta('put')
put_gamma_q8 = model_q8.gamma('put')

print(f"Call Price: {call_price_q8:.4f}")
print(f"Put Price: {put_price_q8:.4f}")
print(f"Call Delta: {call_delta_q8:.4f}")
print(f"Call Gamma: {call_gamma_q8:.4f}")
print(f"Put Delta: {put_delta_q8:.4f}")
print(f"Put Gamma: {put_gamma_q8:.4f}")


Question 8: Jump Intensity λ = 0.75
Call Price: 9.2586
Put Price: 4.3815
Call Delta: 0.7532
Call Gamma: 0.0147
Put Delta: -0.2468
Put Gamma: 0.0147


In [3]:
# Question 9: lambda = 0.25
print("\n" + "=" * 60)
print("Question 9: Jump Intensity λ = 0.25")
print("=" * 60)
model_q9 = MertonJumpModel(S0, K, T, r, sigma, mu_j, delta_j, 0.25)
call_price_q9 = model_q9.price_option('call')
put_price_q9 = model_q9.price_option('put')
call_delta_q9 = model_q9.delta('call')
call_gamma_q9 = model_q9.gamma('call')
put_delta_q9 = model_q9.delta('put')
put_gamma_q9 = model_q9.gamma('put')

print(f"Call Price: {call_price_q9:.4f}")
print(f"Put Price: {put_price_q9:.4f}")
print(f"Call Delta: {call_delta_q9:.4f}")
print(f"Call Gamma: {call_gamma_q9:.4f}")
print(f"Put Delta: {put_delta_q9:.4f}")
print(f"Put Gamma: {put_gamma_q9:.4f}")


Question 9: Jump Intensity λ = 0.25
Call Price: 10.6531
Put Price: 5.7760
Call Delta: 0.6481
Call Gamma: 0.0185
Put Delta: -0.3519
Put Gamma: 0.0185


In [4]:
# Summary Table
print("\n" + "=" * 60)
print("Summary Comparison")
print("=" * 60)
results = pd.DataFrame({
    'Parameter': ['Call Price', 'Put Price', 'Call Delta', 'Call Gamma', 'Put Delta', 'Put Gamma'],
    'λ = 0.75': [call_price_q8, put_price_q8, call_delta_q8, call_gamma_q8, put_delta_q8, put_gamma_q8],
    'λ = 0.25': [call_price_q9, put_price_q9, call_delta_q9, call_gamma_q9, put_delta_q9, put_gamma_q9]
})
print(results.to_string(index=False))


Summary Comparison
 Parameter  λ = 0.75  λ = 0.25
Call Price  9.258599 10.653101
 Put Price  4.381541  5.776044
Call Delta  0.753190  0.648097
Call Gamma  0.014730  0.018457
 Put Delta -0.246810 -0.351903
 Put Gamma  0.014730  0.018457


In [9]:
# Down-and-in Put Option using Merton Model
class MertonBarrierOption(MertonJumpModel):
    def __init__(self, S0, K, T, r, sigma, mu_j, delta_j, lambda_j, barrier):
        super().__init__(S0, K, T, r, sigma, mu_j, delta_j, lambda_j)
        self.barrier = barrier

    def price_di_put(self, n_simulations=50000, n_steps=252):
        """Price down-and-in put with correct jump dynamics"""
        dt = self.T / n_steps
        payoffs = []

        np.random.seed(42)
        for _ in range(n_simulations):
            S = self.S0
            barrier_hit = False

            for step in range(n_steps):
                # Diffusion component
                dW = np.random.normal(0, np.sqrt(dt))
                diffusion_return = (self.r - 0.5 * self.sigma**2) * dt + self.sigma * dW

                # Jump component (Poisson-driven)
                n_jumps = np.random.poisson(self.lambda_j * dt)
                if n_jumps > 0:
                    jump_sizes = np.random.normal(self.mu_j, self.delta_j, n_jumps)
                    jump_return = np.sum(jump_sizes)
                else:
                    jump_return = 0

                # Total return and update price
                total_return = diffusion_return + jump_return
                S = S * np.exp(total_return)

                if S <= self.barrier:
                    barrier_hit = True

            payoff = max(self.K - S, 0) if barrier_hit else 0
            payoffs.append(payoff)

        price = np.exp(-self.r * self.T) * np.mean(payoffs)
        return price


In [10]:

# Question 15: Down-and-in put with barrier
print("\n" + "=" * 60)
print("Down-and-in Put Option (from Question 8 data)")
print("=" * 60)
barrier_level = 65
strike_di = 65

model_di = MertonBarrierOption(S0, strike_di, T, r, sigma, mu_j, delta_j, 0.75, barrier_level)
di_put_price = model_di.price_di_put(n_simulations=50000)

# European put from Question 8 (for comparison, recalculate with strike 65)
model_euro_put = MertonJumpModel(S0, strike_di, T, r, sigma, mu_j, delta_j, 0.75)
euro_put_price = model_euro_put.price_option('put')

print(f"Down-and-in Put Price (K={strike_di}, Barrier={barrier_level}): {di_put_price:.4f}")
print(f"European Put Price (K={strike_di}): {euro_put_price:.4f}")
print(f"Difference: {euro_put_price - di_put_price:.4f}")
print(f"DI Put as % of European Put: {(di_put_price/euro_put_price)*100:.2f}%")


Down-and-in Put Option (from Question 8 data)
Down-and-in Put Price (K=65, Barrier=65): 7.3464
European Put Price (K=65): 0.1992
Difference: -7.1472
DI Put as % of European Put: 3688.64%


Why Down-and-In Put > European Put:
> Extreme jumps guarantee barrier hits: With -50% average jumps, the stock crashes to $65 almost certainly, so the down-and-in condition is nearly always met.


> European put stays worthless: It only pays if stock is below $65 at expiry—but after a crash, the stock often recovers, leaving it worthless at maturity.


> Down-and-in locks in value: Once activated by a crash, it preserves option value even if stock recovers—capturing the downside path that vanilla puts miss.